In [2]:
import numpy as np 
import pandas as pd 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/datasets/mervebzn/hospitaldataset/LUTH_Hospital_Enhanced_Dataset.xlsx


# Initial Exploration

In [3]:
df= pd.read_excel("/kaggle/input/datasets/mervebzn/hospitaldataset/LUTH_Hospital_Enhanced_Dataset.xlsx")

df.shape
df.head()

,Patient ID,Name,Age,Gender,Department,Diagnosis,Treatment,Doctor,Visit Date,Cost (Naira),Ward,Payment Method,Outcome,Follow-Up Required,Satisfaction Score (1-5),Admission Time,Discharge Time
0,LUTH-1000,Patient_1,44,Female,Oncology,Malaria,Surgery,Dr. Obi,2024-02-21,243721.97,Ward D,Cash,Deceased,No,4,2024-02-21 13:13:00,2024-03-02 13:13:00
1,LUTH-1001,Patient_2,51,Male,Neurology,Cancer,Referral,Dr. Bello,2024-11-22,150679.18,Ward C,Insurance,Under Observation,No,1,2024-11-22 17:22:00,2024-12-05 17:22:00
2,LUTH-1002,Patient_3,20,Female,Cardiology,Fracture,Physiotherapy,Dr. Johnson,2024-09-18,97198.81,Ward E,Cash,Deceased,Yes,4,2024-09-18 10:34:00,2024-10-02 10:34:00
3,LUTH-1003,Patient_4,73,Male,Orthopedics,Infection,Physiotherapy,Dr. Okafor,2024-09-14,178102.61,Ward E,Card,Recovered,No,3,2024-09-14 11:34:00,2024-09-25 11:34:00
4,LUTH-1004,Patient_5,78,Male,Pediatrics,Fracture,Surgery,Dr. Musa,2024-07-08,201972.05,Ward E,Insurance,Referred,No,4,2024-07-08 14:52:00,2024-07-23 14:52:00


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 17 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Patient ID                500 non-null    object        
 1   Name                      500 non-null    object        
 2   Age                       500 non-null    int64         
 3   Gender                    500 non-null    object        
 4   Department                500 non-null    object        
 5   Diagnosis                 500 non-null    object        
 6   Treatment                 500 non-null    object        
 7   Doctor                    500 non-null    object        
 8   Visit Date                500 non-null    datetime64[ns]
 9   Cost (Naira)              500 non-null    float64       
 10  Ward                      500 non-null    object        
 11  Payment Method            500 non-null    object        
 12  Outcome               

In [5]:
# Missing values check
df.isnull().sum()

Patient ID                  0
Name                        0
Age                         0
Gender                      0
Department                  0
Diagnosis                   0
Treatment                   0
Doctor                      0
Visit Date                  0
Cost (Naira)                0
Ward                        0
Payment Method              0
Outcome                     0
Follow-Up Required          0
Satisfaction Score (1-5)    0
Admission Time              0
Discharge Time              0
dtype: int64

In [6]:
# Duplicate
df.duplicated().sum()

np.int64(0)

In [7]:
df["Patient ID"].duplicated().sum()

np.int64(0)

# Data Cleaning

In [8]:
# Conver data types
df['Visit Date'] = pd.to_datetime(df['Visit Date'])
df['Admission Time'] = pd.to_datetime(df['Admission Time'])
df['Discharge Time'] = pd.to_datetime(df['Discharge Time'])

df['Patient ID'] = df['Patient ID'].astype(str)


In [9]:
cat_cols = ["Gender", "Department", "Diagnosis", "Treatment", "Doctor",
            "Ward", "Payment Method", "Outcome", "Follow-Up Required"]

for col in cat_cols:
    df[col]= df[col].astype(str).str.strip()
    df[col] = df[col].astype('category')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 17 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Patient ID                500 non-null    object        
 1   Name                      500 non-null    object        
 2   Age                       500 non-null    int64         
 3   Gender                    500 non-null    category      
 4   Department                500 non-null    category      
 5   Diagnosis                 500 non-null    category      
 6   Treatment                 500 non-null    category      
 7   Doctor                    500 non-null    category      
 8   Visit Date                500 non-null    datetime64[ns]
 9   Cost (Naira)              500 non-null    float64       
 10  Ward                      500 non-null    category      
 11  Payment Method            500 non-null    category      
 12  Outcome               

In [10]:
# Standardization check
for col in cat_cols:
    print(col, '-> \n', df[col].unique().tolist())

Gender -> 
 ['Female', 'Male']
Department -> 
 ['Oncology', 'Neurology', 'Cardiology', 'Orthopedics', 'Pediatrics', 'Emergency', 'General Surgery']
Diagnosis -> 
 ['Malaria', 'Cancer', 'Fracture', 'Infection', 'Hypertension', 'Diabetes', 'Stroke']
Treatment -> 
 ['Surgery', 'Referral', 'Physiotherapy', 'Observation', 'Chemotherapy', 'Medication']
Doctor -> 
 ['Dr. Obi', 'Dr. Bello', 'Dr. Johnson', 'Dr. Okafor', 'Dr. Musa', 'Dr. Ahmed', 'Dr. Adeyemi']
Ward -> 
 ['Ward D', 'Ward C', 'Ward E', 'Ward A', 'Ward B']
Payment Method -> 
 ['Cash', 'Insurance', 'Card']
Outcome -> 
 ['Deceased', 'Under Observation', 'Recovered', 'Referred']
Follow-Up Required -> 
 ['No', 'Yes']


In [11]:
# Unrealistic value check
print(df['Age'].describe())
print('\nAge <0  or >120:', ((df['Age']<0) | (df['Age']>120)).sum())

count    500.000000
mean      45.196000
std       25.939377
min        1.000000
25%       23.000000
50%       44.500000
75%       68.000000
max       90.000000
Name: Age, dtype: float64

Age <0  or >120: 0


In [12]:
print(df['Cost (Naira)'].describe())
print('Cost <= 0:', (df['Cost (Naira)'] <=0).sum())

count       500.000000
mean     132347.486460
std       68992.844624
min       10211.400000
25%       75035.492500
50%      130198.560000
75%      188105.530000
max      249943.290000
Name: Cost (Naira), dtype: float64
Cost <= 0: 0


In [13]:
print(df["Satisfaction Score (1-5)"].describe())
print('Unrealistick satisfaction:', ((df['Satisfaction Score (1-5)'] <1) | (df['Satisfaction Score (1-5)']>5)).sum())

count    500.000000
mean       2.986000
std        1.390565
min        1.000000
25%        2.000000
50%        3.000000
75%        4.000000
max        5.000000
Name: Satisfaction Score (1-5), dtype: float64
Unrealistick satisfaction: 0


In [14]:
# Logical date checks
print('Discharge < Admission :', (df["Discharge Time"] < df["Admission Time"]).sum())
print('Visit Date != Admission Date :', (df['Visit Date'].dt.date != df['Admission Time'].dt.date).sum())

Discharge < Admission : 0
Visit Date != Admission Date : 0


In [15]:
# Outlier
for col in ['Age', 'Cost (Naira)']:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3-Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    outliers = df[(df[col]< lower) | (df[col]> upper)]
    print(f"{col}: {len(outliers)} outliers (lower: {lower:.1f}, upper: {upper:.1f})")

Age: 0 outliers (lower: -44.5, upper: 135.5)
Cost (Naira): 0 outliers (lower: -94569.6, upper: 357710.6)


In [16]:
print(df.info())
print(df.describe(include="all").T)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 17 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Patient ID                500 non-null    object        
 1   Name                      500 non-null    object        
 2   Age                       500 non-null    int64         
 3   Gender                    500 non-null    category      
 4   Department                500 non-null    category      
 5   Diagnosis                 500 non-null    category      
 6   Treatment                 500 non-null    category      
 7   Doctor                    500 non-null    category      
 8   Visit Date                500 non-null    datetime64[ns]
 9   Cost (Naira)              500 non-null    float64       
 10  Ward                      500 non-null    category      
 11  Payment Method            500 non-null    category      
 12  Outcome               

# CALCULATED COLUMNS 

In [17]:
# Length of Stay
df['LOS_Days'] = (df['Discharge Time'] - df['Admission Time']).dt.days

print(df['LOS_Days'])

0      10
1      13
2      14
3      11
4      15
       ..
495     1
496     3
497     8
498     8
499     2
Name: LOS_Days, Length: 500, dtype: int64


In [18]:
# Cost per day
effective_days = np.maximum(df['LOS_Days'],1)
df['Cost_per_day']= (df['Cost (Naira)']/effective_days).round(2)
print(df[["Cost (Naira)", "LOS_Days", "Cost_per_day"]].head())

   Cost (Naira)  LOS_Days  Cost_per_day
0     243721.97        10      24372.20
1     150679.18        13      11590.71
2      97198.81        14       6942.77
3     178102.61        11      16191.15
4     201972.05        15      13464.80


In [19]:
print(
    df.groupby(["Department", "Ward"], observed=True)["Cost_per_day"]
    .mean()
    .round(2)
)

Department       Ward  
Cardiology       Ward A    21930.41
                 Ward B    27030.12
                 Ward C    26142.65
                 Ward D    33633.87
                 Ward E    35895.01
Emergency        Ward A    38047.35
                 Ward B    29402.29
                 Ward C    23531.73
                 Ward D    33916.51
                 Ward E    28205.66
General Surgery  Ward A    34893.67
                 Ward B    15120.61
                 Ward C    27250.22
                 Ward D    56494.04
                 Ward E    33336.77
Neurology        Ward A    26176.09
                 Ward B    30572.05
                 Ward C    19704.36
                 Ward D    18384.85
                 Ward E    32154.78
Oncology         Ward A    30441.62
                 Ward B    29385.76
                 Ward C    29108.02
                 Ward D    22874.33
                 Ward E    19418.72
Orthopedics      Ward A    50019.11
                 Ward B    28219.40
    

In [20]:
# Age group
bins = [0,12,18,35,60,120]
labels = ["Child (0-12)", "Teen (13-18)", "Young Adult (19-35)", "Adult (36-60)", "Senior (60+)"]
df['Age_Group'] = pd.cut(df['Age'], bins= bins, labels=labels)

df['Age_Group']

0            Adult (36-60)
1            Adult (36-60)
2      Young Adult (19-35)
3             Senior (60+)
4             Senior (60+)
              ...         
495           Senior (60+)
496           Senior (60+)
497    Young Adult (19-35)
498          Adult (36-60)
499           Senior (60+)
Name: Age_Group, Length: 500, dtype: category
Categories (5, object): ['Child (0-12)' < 'Teen (13-18)' < 'Young Adult (19-35)' < 'Adult (36-60)' < 'Senior (60+)']

In [21]:
# Visit Month / Quarter / Weekday
df['Visit_Month'] = df['Visit Date'].dt.month_name()
df['Visit_Quarter'] = df['Visit Date'].dt.quarter
df['Visit_Weekday'] = df['Visit Date'].dt.day_name()

In [22]:
# Is_Recovered / Is_Deceased flag 
df['Is_Recovered'] = (df['Outcome']=='Recovered').astype(int)
df["Is_Deceased"] = (df["Outcome"] == "Deceased").astype(int)
df['Is_FollowUp'] = (df['Follow-Up Required'] == 'Yes').astype(int)

In [23]:
df['Is_FollowUp']

0      0
1      0
2      1
3      0
4      0
      ..
495    1
496    1
497    1
498    1
499    0
Name: Is_FollowUp, Length: 500, dtype: int64

In [24]:
df.to_csv("LUTH_Hospital_Cleaned.csv", index=False)

## Doctor Dimension 

In [27]:
doctor_dim = df.groupby(['Doctor','Department'], observed=True).agg(
    Total_Patients = ('Patient ID', 'count'),
    Avg_Satisfaction=('Satisfaction Score (1-5)', 'mean'),
    Recovery_Rate= ('Is_Recovered', 'mean'),
    Avg_LOS_Days = ('LOS_Days', 'mean')
).round(2).reset_index()

doctor_dim 

,Doctor,Department,Total_Patients,Avg_Satisfaction,Recovery_Rate,Avg_LOS_Days
0,Dr. Adeyemi,Cardiology,7,2.86,0.43,8.29
1,Dr. Adeyemi,Emergency,14,2.93,0.36,6.14
2,Dr. Adeyemi,General Surgery,14,2.79,0.36,7.79
3,Dr. Adeyemi,Neurology,11,3.27,0.18,8.18
4,Dr. Adeyemi,Oncology,11,3.36,0.27,8.09
5,Dr. Adeyemi,Orthopedics,16,3.19,0.19,7.81
6,Dr. Adeyemi,Pediatrics,10,2.90,0.30,10.50
7,Dr. Ahmed,Cardiology,17,3.12,0.18,6.06
8,Dr. Ahmed,Emergency,12,3.25,0.17,7.58
9,Dr. Ahmed,General Surgery,10,3.40,0.30,9.10


In [28]:
doctor_dim.to_csv("Doctor_Dimension.csv", index=False)